# PancapChain 在 SA-Pancap 上的复现

这个 notebook 用来复现论文 Table 3 最后一行对应的结果：
输入 SA-Pancap 的验证集与测试集图像，使用论文中的 **Pancap-Chain** 模型生成全景描述，再按照论文中的 **Pancap-score** 评分标准进行评测。

本 notebook 的目标是：
- 在 **Colab A100** 环境中运行。
- 数据输入分别来自 `val_images/` 与 `test_images/`。
- 过程中的逐图全景描述结果会保存下来。
- 每个 split 的聚合 caption 结果也会保存下来，便于后续检查或复评。
- 最终输出验证集和测试集在 `Tag / Loc / Att / Rel / Glo / All` 六个指标上的复现结果，并与论文 Table 3 对照。



## Step 1：挂载 Google Drive 并确认 GPU

建议在 Colab 中选择 **A100 GPU** 后再运行本 notebook。
如果你的数据、模型或缓存放在 Drive 中，这一步会把 Drive 挂载到 `/content/drive`。



In [ ]:
from google.colab import drive

drive.mount('/content/drive')

!nvidia-smi



Mounted at /content/drive
Sat Apr 25 10:17:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

## Step 2：克隆 Pancap 仓库

这里直接复用官方仓库，后续推理与 PancapScore 评测都基于仓库内脚本完成。



In [ ]:
%cd /content
!rm -rf Pancap
!git clone https://github.com/Visual-AI/Pancap.git
%cd /content/Pancap
!git rev-parse HEAD



/content
Cloning into 'Pancap'...
remote: Enumerating objects: 182, done.
remote: Counting objects: 100% (182/182), done.
remote: Compressing objects: 100% (138/138), done.
remote: Total 182 (delta 39), reused 169 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (182/182), 15.56 MiB | 18.40 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/Pancap
5cb87b15e33c10180397257c655dcf18642a992f


In [ ]:
!pip uninstall bitsandbytes accelerate

Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Would remove:
    /usr/local/bin/accelerate
    /usr/local/bin/accelerate-config
    /usr/local/bin/accelerate-estimate-memory
    /usr/local/bin/accelerate-launch
    /usr/local/bin/accelerate-merge-weights
    /usr/local/lib/python3.12/dist-packages/accelerate-1.13.0.dist-info/*
    /usr/local/lib/python3.12/dist-packages/accelerate/*
Proceed (Y/n)? Y
  Successfully uninstalled accelerate-1.13.0


In [ ]:
!pip install https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.9.0/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 10.1 MB/s eta 0:00:00


## Step 3：安装依赖并做 Colab 兼容处理

这里采用和 `Pancap_Table3_A100.ipynb` 一样的单环境方式，直接在当前 Colab Python 环境安装依赖，不额外创建虚拟环境。

额外做两件事：
- 放宽部分依赖版本限制，降低 Colab 环境冲突概率。
- 把 PancapScore 里几个大模型加载点改成更适合 A100/Colab 的 4-bit 方式。



In [ ]:
from pathlib import Path
# !pip install transformers==4.43.1 accelerate==1.1.1 bitsandbytes>=0.43.0 sentencepiece protobuf \
#     nltk==3.9.1 sentence-transformers==3.3.1 factualscenegraph==0.5.0 pillow tqdm shortuuid huggingface_hub pandas
# !pip install transformers==4.34.0 accelerate==0.21.0 bitsandbytes>=0.43.0 sentencepiece==0.1.99 protobuf \
#     nltk==3.9.1 sentence-transformers==2.2.1 factualscenegraph==0.5.0 pillow tqdm shortuuid huggingface_hub==0.17.3 pandas
!pip install transformers==4.45.1 accelerate>=0.26.0 bitsandbytes>=0.43.0 sentencepiece>=0.1.99 protobuf \
    nltk==3.9.1 sentence-transformers>=3.0.0 factualscenegraph==0.5.0 pillow tqdm shortuuid huggingface_hub>=0.20.0 pandas

import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')

for rel_path in [
    'pancapscore/extract_content.py',
    'pancapscore/generate_questions.py',
    'pancapscore/answer_questions.py',
]:
    path = Path('/content/Pancap') / rel_path
    src = path.read_text(encoding='utf-8')
    old = '        torch_dtype=torch.bfloat16,\n        attn_implementation="flash_attention_2",\n        device_map={"" : gpu_idx},\n'
    new = '        torch_dtype=torch.bfloat16,\n        device_map={"" : gpu_idx},\n        load_in_4bit=True,\n'
    if old in src:
        src = src.replace(old, new)
        path.write_text(src, encoding='utf-8')
        print(f'已补丁: {rel_path}')
    else:
        print(f'跳过: {rel_path}，未找到预期代码块。')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


跳过: pancapscore/extract_content.py，未找到预期代码块。
跳过: pancapscore/generate_questions.py，未找到预期代码块。
跳过: pancapscore/answer_questions.py，未找到预期代码块。


## Step 4：检查运行环境

确认当前 Python、PyTorch、Transformers 和 GPU 可用状态，避免后面推理时才发现环境不完整。



In [ ]:
import sys
import torch
import transformers

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))



Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Torch: 2.10.0+cu128
Transformers: 4.45.1
CUDA available: True
GPU count: 1
GPU name: NVIDIA A100-SXM4-40GB


## Step 5：配置路径与运行参数

这里把需要你手动确认的路径集中放在一个单元里：
- `VAL_DATA_ROOT`：验证集图像根目录。
- `TEST_DATA_ROOT`：测试集图像根目录。
- `ASM_REPO_ID`：官方基座模型 ASMv2 的 Hugging Face repo id。
- `LORA_REPO_ID`：官方 Pancap-Chain LoRA 的 Hugging Face repo id。
- `RESULT_ROOT`：推理结果、聚合 caption、缓存和评测结果的统一保存目录。

注意：
- 这两个数据目录都应该包含官方 JSON 中的相对路径，例如 `sam/sa_000000/...jpg`。
- notebook 会自动下载 `ASMv2` 和 `PancapChain-lora-13B`，然后自动 merge 成可直接推理的模型。
- 如果你已经提前把模型下载到 Drive，也可以直接改 `ASM_PATH`、`LORA_PATH` 或 `MERGED_PATH`。



In [ ]:
import json
import re
import shutil
import subprocess
import sys

REPO_DIR = Path('/content/Pancap')

VAL_DATA_ROOT = Path('/content/drive/MyDrive/val_images')
TEST_DATA_ROOT = Path('/content/drive/MyDrive/test_images')

RESULT_ROOT = Path('/content/drive/MyDrive/pancapchain_runs')
MODEL_CACHE_ROOT = Path('/content/drive/MyDrive/pancap_models')
MODEL_ALIAS = 'pancapchain-13b'
ASM_REPO_ID = 'OpenGVLab/ASMv2'
LORA_REPO_ID = 'LasNack/PancapChain-lora-13B'
ASM_PATH = MODEL_CACHE_ROOT / 'ASMv2'
LORA_PATH = MODEL_CACHE_ROOT / 'PancapChain-lora-13B'
MERGED_PATH = MODEL_CACHE_ROOT / 'PancapChain-merged'
SPLITS = ('val', 'test')
NUM_GPUS = 1
TEMPERATURE = 0.0
MAX_NEW_TOKENS = 1024
CONV_MODE = 'vicuna_v1'

DATA_ROOTS = {
    'val': VAL_DATA_ROOT,
    'test': TEST_DATA_ROOT,
}

SPLIT_FILES = {
    'val': REPO_DIR / 'playground/data/pancap/sapancap_val_data_list.json',
    'test': REPO_DIR / 'playground/data/pancap/sapancap_test_data_list.json',
}
GT_CAPTION_FILES = {
    'val': REPO_DIR / 'playground/data/pancap/sapancap_val_gtcaption.json',
    'test': REPO_DIR / 'playground/data/pancap/sapancap_test_gtcaption.json',
}
EXPECTED_COUNTS = {'val': 500, 'test': 130}

RAW_PRED_ROOT = RESULT_ROOT / 'generated_pancap_captions'
AGG_ROOT = RESULT_ROOT / 'aggregated_captions'
CACHE_ROOT = RESULT_ROOT / 'pancapscore_cache'
TABLE_ROOT = RESULT_ROOT / 'table3_results'
LOCAL_QUESTION_ROOT = RESULT_ROOT / 'localized_question_files'
LOCALIZATION_MAP_ROOT = RESULT_ROOT / 'image_key_mappings'
SUBSET_ROOT = RESULT_ROOT / 'subset_eval'

for path_obj in [
    RESULT_ROOT,
    MODEL_CACHE_ROOT,
    RAW_PRED_ROOT,
    AGG_ROOT,
    CACHE_ROOT,
    TABLE_ROOT,
    LOCAL_QUESTION_ROOT,
    LOCALIZATION_MAP_ROOT,
    SUBSET_ROOT,
]:
    path_obj.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None, env=None, check=True):
    """运行子进程并在失败时抛出完整日志。"""
    printable = cmd if isinstance(cmd, str) else ' '.join(str(x) for x in cmd)
    print(f'\n[run] {printable}')
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(completed.stdout)
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'命令执行失败，退出码={completed.returncode}\n'
            f'命令: {printable}\n'
            f'完整输出:\n{completed.stdout}'
        )
    return completed


def prediction_dir(split: str) -> Path:
    """返回逐图预测结果目录。"""
    return RAW_PRED_ROOT / split / MODEL_ALIAS


def aggregated_caption_file(split: str) -> Path:
    """返回聚合后的 caption 文件路径。"""
    return AGG_ROOT / split / f'{MODEL_ALIAS}_all_predictions.json'


def localized_question_file(split: str) -> Path:
    """返回适配当前图像目录结构后的 question file 路径。"""
    return LOCAL_QUESTION_ROOT / f'sapancap_{split}_localized.json'


def image_key_mapping_file(split: str) -> Path:
    """保存本地化图片键到官方图片键的映射表。"""
    return LOCALIZATION_MAP_ROOT / f'sapancap_{split}_image_key_map.json'


def ensure_parent(path_obj: Path):
    """确保目标文件的父目录存在。"""
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def resolve_image_path(root: Path, rel_path: str):
    """在官方层级、扁平文件名和仅文件名三种形式里定位图像。"""
    candidates = [
        root / rel_path,
        root / rel_path.replace('/', '_'),
        root / Path(rel_path).name,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def localized_image_relpath(root: Path, rel_path: str):
    """返回当前数据目录下实际存在的相对图片路径。"""
    real_path = resolve_image_path(root, rel_path)
    if real_path is None:
        return None
    return real_path.relative_to(root).as_posix()


def build_localized_question_file(split: str):
    """生成 Colab 可直接读取的 question file，并同步写出图片键映射表。"""
    root = DATA_ROOTS[split]
    src = SPLIT_FILES[split]
    dst = localized_question_file(split)
    mapping_dst = image_key_mapping_file(split)

    with open(src, 'r', encoding='utf-8') as f:
        records = json.load(f)

    patched = []
    missing = []
    image_key_mapping = {}

    for item in records:
        image_rel = item['image'] if isinstance(item, dict) else item
        localized_rel = localized_image_relpath(root, image_rel)
        if localized_rel is None:
            missing.append(image_rel)
            continue

        image_key_mapping[localized_rel] = image_rel

        if isinstance(item, dict):
            obj = dict(item)
            obj['image'] = localized_rel
        else:
            obj = localized_rel
        patched.append(obj)

    if missing:
        preview = '\n'.join(missing[:10])
        print(f'⚠️   {split} 有 {len(missing)} 张图像无法定位，将跳过这些图片。前 10 个缺失项：\n{preview}')

    with open(dst, 'w', encoding='utf-8') as f:
        json.dump(patched, f, ensure_ascii=False, indent=2)
    with open(mapping_dst, 'w', encoding='utf-8') as f:
        json.dump(image_key_mapping, f, ensure_ascii=False, indent=2)

    return dst


print('验证集图像目录:', VAL_DATA_ROOT)
print('测试集图像目录:', TEST_DATA_ROOT)
print('ASMv2 repo id:', ASM_REPO_ID)
print('Pancap-Chain LoRA repo id:', LORA_REPO_ID)
print('模型缓存目录:', MODEL_CACHE_ROOT)
print('结果根目录:', RESULT_ROOT)


验证集图像目录: /content/drive/MyDrive/val_images
测试集图像目录: /content/drive/MyDrive/test_images
ASMv2 repo id: OpenGVLab/ASMv2
Pancap-Chain LoRA repo id: LasNack/PancapChain-lora-13B
模型缓存目录: /content/drive/MyDrive/pancap_models
结果根目录: /content/drive/MyDrive/pancapchain_runs


## Step 6：检查验证集与测试集图像

这里会读取官方的 `sapancap_val_data_list.json` 和 `sapancap_test_data_list.json`，
逐条检查 `val_images/` 与 `test_images/` 下是否存在对应图片。



In [ ]:
LOCALIZED_SPLIT_FILES = {}

for split in SPLITS:
    data_root = DATA_ROOTS[split]
    assert data_root.exists(), f'{split} 图像根目录不存在: {data_root}'

    with open(SPLIT_FILES[split], 'r', encoding='utf-8') as f:
        image_list = json.load(f)

    print(f'[{split}] 官方样本数 = {len(image_list)}')
    assert len(image_list) == EXPECTED_COUNTS[split], f'{split} 数量异常: {len(image_list)}'

    missing = []
    matched_nested = 0
    matched_flattened = 0
    matched_basename = 0

    for item in image_list:
        rel_path = item['image'] if isinstance(item, dict) else item

        p1 = data_root / rel_path
        p2 = data_root / rel_path.replace('/', '_')
        p3 = data_root / Path(rel_path).name

        if p1.exists():
            matched_nested += 1
        elif p2.exists():
            matched_flattened += 1
        elif p3.exists():
            matched_basename += 1
        else:
            missing.append(rel_path)

    print(f'[{split}] 匹配到官方层级格式: {matched_nested}')
    print(f'[{split}] 匹配到扁平文件名格式: {matched_flattened}')
    print(f'[{split}] 匹配到仅文件名格式: {matched_basename}')

    if missing:
        preview = '\n'.join(missing[:10])
        # raise FileNotFoundError(
        #     f'{split} 仍有 {len(missing)} 张图像无法定位。前 10 个缺失文件：\n{preview}'
        # )
        print(f'⚠️   {split} 有 {len(missing)} 张图像无法定位，将跳过这些图片。前 10 个缺失项：\n{preview}')

    LOCALIZED_SPLIT_FILES[split] = build_localized_question_file(split)
    print(f'[{split}] 已生成本地兼容版 question file: {LOCALIZED_SPLIT_FILES[split]}')

print('验证集与测试集图像都已完成定位，并生成了兼容当前目录结构的 question file。')

[val] 官方样本数 = 500
[val] 匹配到官方层级格式: 0
[val] 匹配到扁平文件名格式: 495
[val] 匹配到仅文件名格式: 0
⚠️   val 有 5 张图像无法定位，将跳过这些图片。前 10 个缺失项：
sam/sa_000000/sa_9139.jpg
sam/sa_000000/sa_3902.jpg
sam/sa_000000/sa_1931.jpg
sam/sa_000003/sa_44690.jpg
sam/sa_000002/sa_32538.jpg
⚠️   val 有 5 张图像无法定位，将跳过这些图片。前 10 个缺失项：
sam/sa_000000/sa_9139.jpg
sam/sa_000000/sa_3902.jpg
sam/sa_000000/sa_1931.jpg
sam/sa_000003/sa_44690.jpg
sam/sa_000002/sa_32538.jpg
[val] 已生成本地兼容版 question file: /content/drive/MyDrive/pancapchain_runs/localized_question_files/sapancap_val_localized.json
[test] 官方样本数 = 130
[test] 匹配到官方层级格式: 0
[test] 匹配到扁平文件名格式: 130
[test] 匹配到仅文件名格式: 0
[test] 已生成本地兼容版 question file: /content/drive/MyDrive/pancapchain_runs/localized_question_files/sapancap_test_localized.json
验证集与测试集图像都已完成定位，并生成了兼容当前目录结构的 question file。


## Step 7：自动下载 Pancap-Chain 模型

这一步会按官方仓库提供的 Hugging Face 权重自动完成模型准备：
- 下载基座模型 `OpenGVLab/ASMv2`。
- 下载官方 `LasNack/PancapChain-lora-13B` 权重。

注意：
- 结合当前实际仓库内容，`LasNack/PancapChain-lora-13B` 下载下来是完整模型格式。
- 因此这里不再执行 LoRA merge，后续推理直接使用该目录作为 `MODEL_PATH`。


In [ ]:
from huggingface_hub import snapshot_download

if not (ASM_PATH / 'config.json').exists():
    print('开始下载 ASMv2 基座模型...')
    snapshot_download(
        repo_id=ASM_REPO_ID,
        local_dir=str(ASM_PATH),
    )
else:
    print('ASMv2 已存在，跳过下载。')

if not (LORA_PATH / 'config.json').exists():
    print('开始下载 Pancap-Chain 权重...')
    snapshot_download(
        repo_id=LORA_REPO_ID,
        local_dir=str(LORA_PATH),
    )
else:
    print('Pancap-Chain 权重已存在，跳过下载。')

MODEL_PATH = LORA_PATH

print('最终使用的模型路径:', MODEL_PATH)
assert MODEL_PATH.exists(), f'模型路径不存在: {MODEL_PATH}'


ASMv2 已存在，跳过下载。
Pancap-Chain 权重已存在，跳过下载。
最终使用的模型路径: /content/drive/MyDrive/pancap_models/PancapChain-lora-13B


## Step 8：在验证集上运行 Pancap-Chain，并保存逐图全景描述

这一阶段会：
- 对验证集图片逐张生成 Pancap-Chain 输出。
- 保存原始逐图结果文件。
- 把逐图结果聚合成一个总的 caption JSON，供 PancapScore 使用。



In [ ]:
import os


def prediction_file(split: str) -> Path:
    """返回逐图预测目录；官方脚本会把每张图的结果单独写到这个目录。"""
    out_dir = RAW_PRED_ROOT / split / MODEL_ALIAS
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir


def remap_aggregated_prediction_keys(split: str, agg_file: Path) -> Path:
    """把聚合预测里的本地化图片键恢复成官方 image_path，供 PancapScore 对齐。"""
    mapping_path = image_key_mapping_file(split)
    with open(mapping_path, 'r', encoding='utf-8') as f:
        image_key_mapping = json.load(f)
    with open(agg_file, 'r', encoding='utf-8') as f:
        records = json.load(f)

    remapped_records = []
    remapped_count = 0
    missing_keys = []

    for record in records:
        obj = dict(record)
        local_key = obj.get('image_path') or obj.get('image')
        if not isinstance(local_key, str):
            remapped_records.append(obj)
            continue

        official_key = image_key_mapping.get(local_key)
        if official_key is None:
            missing_keys.append(local_key)
            official_key = local_key
        else:
            remapped_count += 1

        if 'image_path' in obj:
            obj['image_path'] = official_key
        if 'image' in obj:
            obj['image'] = official_key
        remapped_records.append(obj)

    with open(agg_file, 'w', encoding='utf-8') as f:
        json.dump(remapped_records, f, ensure_ascii=False, indent=2)

    print(f'[{split}] 已将 {remapped_count} 条聚合预测键恢复为官方 image_path。')
    if missing_keys:
        preview = '\n'.join(sorted(set(missing_keys))[:10])
        print(f'⚠️ [{split}] 有 {len(set(missing_keys))} 个聚合键未命中映射表。前 10 个键：\n{preview}')

    return agg_file


def return_file(split: str):
    """复用已生成的逐图结果，并重新整理聚合文件与官方图片键。"""
    out_dir = prediction_file(split)
    agg_file = aggregated_caption_file(split)
    ensure_parent(agg_file)
    if agg_file.exists():
        agg_file.unlink()

    generated_path = Path(f"{out_dir}-all-predictions.json")
    if generated_path.exists():
        shutil.copyfile(generated_path, agg_file)
    else:
        fallback_inside = out_dir / f"{out_dir.name}-all-predictions.json"
        if fallback_inside.exists():
            shutil.copyfile(fallback_inside, agg_file)
        else:
            raise FileNotFoundError(f'没有找到聚合后的 caption 文件: {generated_path} 或 {fallback_inside}')

    remap_aggregated_prediction_keys(split, agg_file)

    print(f'[{split}] 逐图结果目录: {out_dir}')
    print(f'[{split}] 聚合 caption 文件: {agg_file}')
    return out_dir, agg_file


def run_inference(split: str):
    """执行 Pancap-Chain 推理，并将聚合结果回写成官方图片键。"""
    out_file = prediction_file(split)

    cmd = [
        sys.executable,
        '-m', 'llava.eval.model_vqa_loader_pancapchain',
        '--model-path', str(LORA_PATH),
        '--question-file', str(LOCALIZED_SPLIT_FILES[split]),
        '--image-folder', str(DATA_ROOTS[split]),
        '--answers-file', str(out_file),
        '--num-chunk', str(NUM_GPUS),
        '--chunk-idx', '0',
        '--temperature', str(TEMPERATURE),
        '--max_new_tokens', str(MAX_NEW_TOKENS),
        '--conv-mode', CONV_MODE,
    ]

    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR) + (':' + env['PYTHONPATH'] if 'PYTHONPATH' in env else '')

    run(cmd, cwd=REPO_DIR, env=env)

    agg_file = aggregated_caption_file(split)
    ensure_parent(agg_file)
    if agg_file.exists():
        agg_file.unlink()

    run([
        sys.executable,
        'scripts_llmjudge/agg_jsons_asv2togt.py',
        '--data-root', str(DATA_ROOTS[split]),
        '--res-dir', str(out_file),
    ], cwd=REPO_DIR, env=env)

    generated_default = Path(str(out_file).replace('.jsonl', '-all-predictions.json'))
    if generated_default.exists():
        shutil.copyfile(generated_default, agg_file)
    else:
        fallback = Path(str(out_file) + '-all-predictions.json')
        if not fallback.exists():
            raise FileNotFoundError(f'没有找到聚合后的 caption 文件: {generated_default} 或 {fallback}')
        shutil.copyfile(fallback, agg_file)

    remap_aggregated_prediction_keys(split, agg_file)

    print(f'[{split}] 逐图结果文件: {out_file}')
    print(f'[{split}] 聚合 caption 文件: {agg_file}')
    return out_file, agg_file


In [ ]:
# from pathlib import Path

# # 1) llava_llama.py: 修 transformers 新版下 llava 重复注册
# llava_llama = Path('/content/Pancap/llava/model/language_model/llava_llama.py')
# lines = llava_llama.read_text(encoding='utf-8').splitlines()

# patched = []
# for line in lines:
#     s = line.strip()

#     if s == 'AutoConfig.register("llava", LlavaConfig)':
#         patched += [
#             'try:',
#             '    AutoConfig.register("llava", LlavaConfig, exist_ok=True)',
#             'except TypeError:',
#             '    try:',
#             '        AutoConfig.register("llava", LlavaConfig)',
#             '    except ValueError:',
#             '        pass',
#             'except ValueError:',
#             '    pass',
#         ]
#         continue

#     if s == 'AutoModelForCausalLM.register(LlavaConfig, LlavaLlamaForCausalLM)':
#         patched += [
#             'try:',
#             '    AutoModelForCausalLM.register(LlavaConfig, LlavaLlamaForCausalLM, exist_ok=True)',
#             'except TypeError:',
#             '    try:',
#             '        AutoModelForCausalLM.register(LlavaConfig, LlavaLlamaForCausalLM)',
#             '    except ValueError:',
#             '        pass',
#             'except ValueError:',
#             '    pass',
#         ]
#         continue

#     patched.append(line)

# llava_llama.write_text('\n'.join(patched) + '\n', encoding='utf-8')
# print('patched:', llava_llama)

# # 2) llava/model/__init__.py: 让 MPT 变成可选导入
# model_init = Path('/content/Pancap/llava/model/__init__.py')
# model_init.write_text(
# """from .language_model.llava_llama import LlavaLlamaForCausalLM, LlavaConfig

# try:
#     from .language_model.llava_mpt import LlavaMPTForCausalLM, LlavaMPTConfig
# except Exception:
#     LlavaMPTForCausalLM = None
#     LlavaMPTConfig = None
# """,
#     encoding='utf-8'
# )
# print('patched:', model_init)

# # 3) builder.py: 给 offload_folder，避免 device_map offload 报错
builder = Path('/content/Pancap/llava/model/builder.py')
orig_text = builder.read_text(encoding='utf-8')

# Patch 1: Add offload_folder
orig_text = orig_text.replace(
    'kwargs = {"device_map": device_map, **kwargs}',
    'kwargs = {"device_map": device_map, "offload_folder": "/content/offload", **kwargs}'
)

# Patch 2: Fix local path loading - add local check before HF Hub download
orig_text = orig_text.replace(
'''            print('Loading additional LLaVA weights...')
            if os.path.exists(os.path.join(model_path, 'non_lora_trainables.bin')):
                non_lora_trainables = torch.load(os.path.join(model_path, 'non_lora_trainables.bin'), map_location='cpu')
            else:
                # this is probably from HF Hub
                from huggingface_hub import hf_hub_download
                def load_from_hf(repo_id, filename, subfolder=None):
                    cache_file = hf_hub_download(
                        repo_id=repo_id,
                        filename=filename,
                        subfolder=subfolder)
                    return torch.load(cache_file, map_location='cpu')
                non_lora_trainables = load_from_hf(model_path, 'non_lora_trainables.bin')''',
'''            print('Loading additional LLaVA weights...')
            local_path = os.path.join(model_path, 'non_lora_trainables.bin')
            if os.path.exists(local_path):
                print(f'Loading non_lora_trainables.bin from local path: {local_path}')
                non_lora_trainables = torch.load(local_path, map_location='cpu')
            else:
                # this is probably from HF Hub
                from huggingface_hub import hf_hub_download
                def load_from_hf(repo_id, filename, subfolder=None):
                    cache_file = hf_hub_download(
                        repo_id=repo_id,
                        filename=filename,
                        subfolder=subfolder)
                    return torch.load(cache_file, map_location='cpu')
                non_lora_trainables = load_from_hf(model_path, 'non_lora_trainables.bin')'''
)

builder.write_text(orig_text, encoding='utf-8')
Path('/content/offload').mkdir(parents=True, exist_ok=True)
print('✅ Fully patched builder.py:')
print('   - Added offload_folder="/content/offload"')
print('   - Added local path check for non_lora_trainables.bin')

# builder.write_text(text, encoding='utf-8')
# Path('/content/offload').mkdir(parents=True, exist_ok=True)
# print('patched:', builder)

✅ Fully patched builder.py:
   - Added offload_folder="/content/offload"
   - Added local path check for non_lora_trainables.bin


In [ ]:
print(Path('/content/Pancap/llava/model/__init__.py').read_text(encoding='utf-8'))

from .language_model.llava_llama import LlavaLlamaForCausalLM, LlavaConfig
from .language_model.llava_mpt import LlavaMPTForCausalLM, LlavaMPTConfig



In [ ]:
VAL_RAW_DIR, VAL_AGG_FILE = run_inference('val')


[run] /usr/bin/python3 -m llava.eval.model_vqa_loader_pancapchain --model-path /content/drive/MyDrive/pancap_models/PancapChain-lora-13B --question-file /content/drive/MyDrive/pancapchain_runs/localized_question_files/sapancap_val_localized.json --image-folder /content/drive/MyDrive/val_images --answers-file /content/drive/MyDrive/pancapchain_runs/generated_pancap_captions/val/pancapchain-13b --num-chunk 1 --chunk-idx 0 --temperature 0.0 --max_new_tokens 1024 --conv-mode vicuna_v1


In [ ]:
VAL_RAW_DIR, VAL_AGG_FILE = return_file('val')

[val] 已将 495 条聚合预测键恢复为官方 image_path。
[val] 逐图结果目录: /content/drive/MyDrive/pancapchain_runs/generated_pancap_captions/val/pancapchain-13b
[val] 聚合 caption 文件: /content/drive/MyDrive/pancapchain_runs/aggregated_captions/val/pancapchain-13b_all_predictions.json


## Step 9：在测试集上运行 Pancap-Chain，并保存逐图全景描述

这一步与上一步相同，只是数据换成测试集。



In [ ]:
TEST_RAW_DIR, TEST_AGG_FILE = run_inference('test')



In [ ]:
TEST_RAW_DIR, TEST_AGG_FILE = return_file('test')


[test] 已将 130 条聚合预测键恢复为官方 image_path。
[test] 逐图结果目录: /content/drive/MyDrive/pancapchain_runs/generated_pancap_captions/test/pancapchain-13b
[test] 聚合 caption 文件: /content/drive/MyDrive/pancapchain_runs/aggregated_captions/test/pancapchain-13b_all_predictions.json


## Step 10：运行验证集 PancapScore 评测

这里按论文对应的 PancapScore 流程执行：
- 先抽取 GT caption 的结构化内容与问题。
- 再抽取预测 caption 的结构化内容。
- 最后运行 Tag / Loc / Att / Rel / Glo / All 六个维度的综合评测。



In [ ]:
import json
import random

SUBSET_SAMPLE_SIZES = {'val': 70, 'test': 50}


def subset_caption_file(split: str, role: str, sample_size: int) -> Path:
    """返回当前 split 的子集 GT 或预测文件路径。"""
    subset_dir = SUBSET_ROOT / split
    subset_dir.mkdir(parents=True, exist_ok=True)
    return subset_dir / f'sapancap_{split}_{role}_subset_{sample_size}.json'


def get_image_key(record: dict):
    """从 GT 或预测记录中提取统一的官方图片键。"""
    for key in ('image_path', 'image', 'id'):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.replace('\\', '/').strip()
    return None


def sample_subset_files(split: str, sample_size: int, gt_path, pred_path):
    """按 image_path 对齐 GT 与预测，并写出独立子集文件，避免覆盖原始文件。"""
    print(f'正在为 {split} 构建 {sample_size} 条样本的评测子集...')

    with open(gt_path, 'r', encoding='utf-8') as f:
        gt_data = json.load(f)
    with open(pred_path, 'r', encoding='utf-8') as f:
        pred_data = json.load(f)

    assert isinstance(gt_data, list) and isinstance(pred_data, list), 'GT 和预测文件都必须是列表格式。'

    gt_dict = {}
    pred_dict = {}

    for item in gt_data:
        key = get_image_key(item)
        if key is not None:
            gt_dict[key] = item

    for item in pred_data:
        key = get_image_key(item)
        if key is not None:
            pred_dict[key] = item

    common_keys = sorted(set(gt_dict.keys()).intersection(pred_dict.keys()))
    if not common_keys:
        raise ValueError(f'{split} 的 GT 与预测没有可对齐的公共 image_path，请先检查聚合预测键是否已恢复为官方格式。')

    rng = random.Random(42)
    sampled_keys = rng.sample(common_keys, min(sample_size, len(common_keys)))

    gt_sampled = [gt_dict[key] for key in sampled_keys]
    pred_sampled = [pred_dict[key] for key in sampled_keys]

    gt_subset_path = subset_caption_file(split, 'gtcaption', len(gt_sampled))
    pred_subset_path = subset_caption_file(split, f'{MODEL_ALIAS}_predictions', len(pred_sampled))

    with open(gt_subset_path, 'w', encoding='utf-8') as f:
        json.dump(gt_sampled, f, ensure_ascii=False, indent=2)
    with open(pred_subset_path, 'w', encoding='utf-8') as f:
        json.dump(pred_sampled, f, ensure_ascii=False, indent=2)

    print(f'[{split}] 原始 GT 数量: {len(gt_data)}，原始预测数量: {len(pred_data)}，可对齐数量: {len(common_keys)}')
    print(f'[{split}] 子集 GT 文件: {gt_subset_path}')
    print(f'[{split}] 子集预测文件: {pred_subset_path}')
    print(f'✅ {split} 集子集构建完成：GT 和预测各保留 {len(gt_sampled)} 条。\n')

    return gt_subset_path, pred_subset_path


VAL_SUBSET_GT_FILE, VAL_SUBSET_PRED_FILE = sample_subset_files(
    'val',
    SUBSET_SAMPLE_SIZES['val'],
    GT_CAPTION_FILES['val'],
    VAL_AGG_FILE,
)
TEST_SUBSET_GT_FILE, TEST_SUBSET_PRED_FILE = sample_subset_files(
    'test',
    SUBSET_SAMPLE_SIZES['test'],
    GT_CAPTION_FILES['test'],
    TEST_AGG_FILE,
)


正在为 val 构建 70 条样本的评测子集...
[val] 原始 GT 数量: 500，原始预测数量: 495，可对齐数量: 495
[val] 子集 GT 文件: /content/drive/MyDrive/pancapchain_runs/subset_eval/val/sapancap_val_gtcaption_subset_70.json
[val] 子集预测文件: /content/drive/MyDrive/pancapchain_runs/subset_eval/val/sapancap_val_pancapchain-13b_predictions_subset_70.json
✅ val 集子集构建完成：GT 和预测各保留 70 条。

正在为 test 构建 50 条样本的评测子集...
[test] 原始 GT 数量: 130，原始预测数量: 130，可对齐数量: 130
[test] 子集 GT 文件: /content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_gtcaption_subset_50.json
[test] 子集预测文件: /content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_pancapchain-13b_predictions_subset_50.json
✅ test 集子集构建完成：GT 和预测各保留 50 条。



In [ ]:
import pandas as pd


def parse_metric_stdout(stdout: str):
    """从 PancapScore 输出中解析六个指标。"""
    patterns = {
        'Tag': r'Tagging F1-Score:\s*([0-9.]+)',
        'Loc': r'Localization F1-Score:\s*([0-9.]+)',
        'Att': r'Attribute F1-Score:\s*([0-9.]+)',
        'Rel': r'Relation F1-Score:\s*([0-9.]+)',
        'Glo': r'Global F1-Score:\s*([0-9.]+)',
        'All': r'Overall PancapScore:\s*([0-9.]+)',
    }
    result = {}
    for key, pattern in patterns.items():
        match = re.search(pattern, stdout)
        if not match:
            raise ValueError(f'无法从输出中解析 {key}。完整输出如下：\n{stdout}')
        result[key] = float(match.group(1)) * 100.0
    return result


def run_pancapscore(split: str, gt_caption: Path, pred_caption: Path, cache_tag: str = 'full'):
    """对指定 GT/预测文件运行完整 PancapScore 流程，并按 cache_tag 隔离缓存。"""
    cache_prefix = f'sapancap_{split}_{cache_tag}'

    gt_content = CACHE_ROOT / 'GT' / 'content' / f'{cache_prefix}_content.json'
    gt_questions = CACHE_ROOT / 'GT' / 'questions' / f'{cache_prefix}_question.json'
    pred_content = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_content.json'
    gt4pred_questions = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_gt4pred_question.json'
    pred_answers = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_pred_answer.json'
    pred_questions = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_pred_question.json'
    pred4gt_questions = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_pred4gt_question.json'
    gt_answers = CACHE_ROOT / MODEL_ALIAS / 'content' / f'{cache_prefix}_gt_answer.json'

    for p in [gt_content, gt_questions, pred_content, gt4pred_questions, pred_answers, pred_questions, pred4gt_questions, gt_answers]:
        p.parent.mkdir(parents=True, exist_ok=True)

    if not gt_content.exists():
        run([
            sys.executable, 'pancapscore/extract_content.py',
            '--num-gpu', '1',
            '--response_key', 'gt_response',
            '--caption_json', str(gt_caption),
            '--content_json', str(gt_content),
        ], cwd=REPO_DIR)

    if not gt_questions.exists():
        run([
            sys.executable, 'pancapscore/generate_questions.py',
            '--num-gpu', '1',
            '--response_key', 'gt_response',
            '--caption_json', str(gt_caption),
            '--content_json', str(gt_content),
            '--question_json', str(gt_questions),
        ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/extract_content.py',
        '--num-gpu', '1',
        '--response_key', 'model_response',
        '--caption_json', str(pred_caption),
        '--content_json', str(pred_content),
    ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/eval_tag_and_loc.py',
        '--cand_file', str(pred_content),
        '--gt_file', str(gt_questions),
        '--save_file', str(gt4pred_questions),
    ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/answer_questions.py',
        '--num-gpu', '1',
        '--response_key', 'model_response',
        '--caption_json', str(pred_caption),
        '--question_json', str(gt4pred_questions),
        '--result_json', str(pred_answers),
    ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/generate_questions.py',
        '--num-gpu', '1',
        '--response_key', 'model_response',
        '--caption_json', str(pred_caption),
        '--content_json', str(pred_content),
        '--question_json', str(pred_questions),
    ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/eval_tag_and_loc.py',
        '--cand_file', str(gt_content),
        '--gt_file', str(pred_questions),
        '--save_file', str(pred4gt_questions),
    ], cwd=REPO_DIR)

    run([
        sys.executable, 'pancapscore/answer_questions.py',
        '--num-gpu', '1',
        '--response_key', 'gt_response',
        '--caption_json', str(gt_caption),
        '--question_json', str(pred4gt_questions),
        '--result_json', str(gt_answers),
    ], cwd=REPO_DIR)

    completed = run([
        sys.executable, 'pancapscore/eval_alldims.py',
        '--cand_file', str(pred_content),
        '--gt_file', str(gt_questions),
        '--gt4pred_answer_file', str(pred_answers),
        '--pred4gt_answer_file', str(gt_answers),
    ], cwd=REPO_DIR)

    result = parse_metric_stdout(completed.stdout)
    result['saved_captions'] = str(pred_caption)
    result['saved_gt_captions'] = str(gt_caption)
    return result


val_result = run_pancapscore('val', VAL_SUBSET_GT_FILE, VAL_SUBSET_PRED_FILE, cache_tag='subset70')
val_result



[run] /usr/bin/python3 pancapscore/extract_content.py --num-gpu 1 --response_key model_response --caption_json /content/drive/MyDrive/pancapchain_runs/subset_eval/val/sapancap_val_pancapchain-13b_predictions_subset_70.json --content_json /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/pancapchain-13b/content/sapancap_val_subset70_content.json
Extracting all content...


Loading checkpoint shards: 100%|██████████| 8/8 [00:09<00:00,  1.25s/it]

100%|██████████| 70/70 [00:00<00:00, 6422.15it/s]
All processes start...
All processes finished!


[run] /usr/bin/python3 pancapscore/eval_tag_and_loc.py --cand_file /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/pancapchain-13b/content/sapancap_val_subset70_content.json --gt_file /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/GT/questions/sapancap_val_subset70_question.json --save_file /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/pancapchain-13b/content/sapancap_val_subset70_gt4pred_question.json
2026

{'Tag': 58.73380984575532,
 'Loc': 26.230759694381984,
 'Att': 46.639494514281104,
 'Rel': 36.35950449625495,
 'Glo': 84.50430703826488,
 'All': 176.41399925449983,
 'saved_captions': '/content/drive/MyDrive/pancapchain_runs/subset_eval/val/sapancap_val_pancapchain-13b_predictions_subset_70.json',
 'saved_gt_captions': '/content/drive/MyDrive/pancapchain_runs/subset_eval/val/sapancap_val_gtcaption_subset_70.json'}

## Step 11：运行测试集 PancapScore 评测

和验证集评测流程相同，这一步输出测试集的六项指标。



In [ ]:
test_result = run_pancapscore('test', TEST_SUBSET_GT_FILE, TEST_SUBSET_PRED_FILE, cache_tag='subset50')
test_result


[run] /usr/bin/python3 pancapscore/extract_content.py --num-gpu 1 --response_key gt_response --caption_json /content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_gtcaption_subset_50.json --content_json /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/GT/content/sapancap_test_subset50_content.json
Extracting all content...

Loading checkpoint shards: 100%|██████████| 8/8 [00:09<00:00,  1.21s/it]

  0%|          | 0/50 [00:00<?, ?it/s]Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)

100%|██████████| 50/50 [1:14:33<00:00, 89.48s/it]
All processes start...
All processes finished!


[run] /usr/bin/python3 pancapscore/generate_questions.py --num-gpu 1 --response_key gt_response --caption_json /content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_gtcaption_subset_50.json --content_json /content/drive/MyDrive/pancapchain_runs/pancapscore_cache/GT/content/sapancap

{'Tag': 58.50995680588088,
 'Loc': 29.700728872091105,
 'Att': 44.43331274979545,
 'Rel': 26.662936573159048,
 'Glo': 86.58256752993596,
 'All': 167.9651917539201,
 'saved_captions': '/content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_pancapchain-13b_predictions_subset_50.json',
 'saved_gt_captions': '/content/drive/MyDrive/pancapchain_runs/subset_eval/test/sapancap_test_gtcaption_subset_50.json'}

## Step 12：整理结果并与论文 Table 3 对照

这里会同时展示：
- 你本次复现得到的验证集与测试集结果。
- 论文 Table 3 中 Pancap-Chain 对应的目标数值。
- 每个指标相对论文数值的差值。
- 生成 caption 的保存路径。



In [ ]:
paper_targets = {
    'val':  {'Tag': 57.56, 'Loc': 30.34, 'Att': 44.78, 'Rel': 34.61, 'Glo': 84.59, 'All': 175.75},
    'test': {'Tag': 56.45, 'Loc': 31.76, 'Att': 44.46, 'Rel': 32.54, 'Glo': 79.85, 'All': 173.19},
}

rows = []
for split, result in [('val', val_result), ('test', test_result)]:
    row = {
        'split': split,
        'saved_captions': result['saved_captions'],
    }
    for key in ['Tag', 'Loc', 'Att', 'Rel', 'Glo', 'All']:
        row[f'{key}_reproduced'] = round(result[key], 2)
        row[f'{key}_paper'] = paper_targets[split][key]
        row[f'{key}_delta'] = round(result[key] - paper_targets[split][key], 2)
    rows.append(row)

result_df = pd.DataFrame(rows)
result_csv = TABLE_ROOT / f'{MODEL_ALIAS}_table3_compare.csv'
result_df.to_csv(result_csv, index=False, encoding='utf-8-sig')

display(result_df)
print('\n结果表已保存到:', result_csv)
print('验证集聚合 caption:', VAL_AGG_FILE)
print('测试集聚合 caption:', TEST_AGG_FILE)
print('验证集逐图结果目录:', VAL_RAW_DIR)
print('测试集逐图结果目录:', TEST_RAW_DIR)



,split,saved_captions,Tag_reproduced,Tag_paper,Tag_delta,Loc_reproduced,Loc_paper,Loc_delta,Att_reproduced,Att_paper,Att_delta,Rel_reproduced,Rel_paper,Rel_delta,Glo_reproduced,Glo_paper,Glo_delta,All_reproduced,All_paper,All_delta
0,val,/content/drive/MyDrive/pancapchain_runs/subset...,58.73,57.56,1.17,26.23,30.34,-4.11,46.64,44.78,1.86,36.36,34.61,1.75,84.50,84.59,-0.09,176.41,175.75,0.66
1,test,/content/drive/MyDrive/pancapchain_runs/subset...,58.51,56.45,2.06,29.70,31.76,-2.06,44.43,44.46,-0.03,26.66,32.54,-5.88,86.58,79.85,6.73,167.97,173.19,-5.22



结果表已保存到: /content/drive/MyDrive/pancapchain_runs/table3_results/pancapchain-13b_table3_compare.csv
验证集聚合 caption: /content/drive/MyDrive/pancapchain_runs/aggregated_captions/val/pancapchain-13b_all_predictions.json
测试集聚合 caption: /content/drive/MyDrive/pancapchain_runs/aggregated_captions/test/pancapchain-13b_all_predictions.json
验证集逐图结果目录: /content/drive/MyDrive/pancapchain_runs/generated_pancap_captions/val/pancapchain-13b
测试集逐图结果目录: /content/drive/MyDrive/pancapchain_runs/generated_pancap_captions/test/pancapchain-13b


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=result_df)

MessageError: Error: credential propagation was unsuccessful